# Display features

In [ ]:
import pickle
with open(r"C:\Users\bilel.guetarni\Desktop\ARTIX\data\artix.pkl", "rb") as f:
    patients = pickle.load(f)

In [ ]:
print(patients[0].clinical.keys())
print(patients[0].id)
print(patients[0].clinical['SEX'])
print(patients[0].clinical['AGE'])
print(patients[0].clinical['ST_STAG'])
print(patients[0].clinical['ECOG'])

In [ ]:
import pandas
pandas.DataFrame(patients[0].clinical_measurements)

In [ ]:
import os
path_ = r"C:\Users\bilel.guetarni\Desktop\ARTIX\features\original\artix"
patient_folder = os.path.join(path_, [i for i in os.listdir(path_) if int(i) == int(patients[0].id)][0])
print(patient_folder)

In [ ]:
radiomics = pandas.read_csv(os.path.join(patient_folder, "deepnn.csv"))
radiomics["id"] = int(patients[0].id)
radiomics

In [ ]:
def load_features(path_, type_, patient_id, columns_to_keep=["oar", "name", "value"]):
    """
    load features

    args:
        path_ (str) path to features folder
        type_ (str) features type (radiomics, dosiomics, dvh, deepnn)
        patient_id (int,str) id of patient to find its folder
        columns_to_keep (list(str)) list of columns name to keep
    """
    patient_folder = os.path.join(path_, [i for i in os.listdir(path_) if int(i) == int(patient_id)][0])
    features = pandas.read_csv(os.path.join(patient_folder, f"{type_}.csv"))
    
    if "Unnamed: 0" in features.columns:
        features = features.drop(columns=["Unnamed: 0"])
    
    if columns_to_keep:
        features = features[columns_to_keep]
    
    return features

def load_radiomics(path_, patient_id, radiomics=["shape", "firstorder", "glcm", "gldm", "glrlm", "glszm", "ngtdm"]):
    df = load_features(path_, "radiomics", patient_id, columns_to_keep=None)
    df = df[(df["type"] == "original") & (df["class"].isin(radiomics))]
    return df.drop(columns=["type"])

def load_dosiomics(path_, patient_id, radiomics=["shape", "firstorder", "glcm", "gldm", "glrlm", "glszm", "ngtdm"]):
    df = load_features(path_, "dosiomics", patient_id, columns_to_keep=None)
    df = df[(df["type"] == "original") & (df["class"].isin(radiomics))]
    return df.drop(columns=["type"])

In [ ]:
df = load_radiomics(path_, patients[0].id)
df

In [ ]:
df = load_dosiomics(path_, patients[0].id)
df

In [ ]:
df = load_features(path_, "dvh", patients[0].id)
df

In [ ]:
df = load_features(path_, "deepnn", patients[0].id)
df

# Fit

In [ ]:
import plotly.express as px
from sklearn.decomposition import PCA

def pca_visualization(x, y):
    x = PCA(n_components=2, random_state=0).fit_transform(x)
    fig = px.scatter(x=x[:,0], y=x[:,1], color=y)
    fig.show()

In [19]:
import pickle
import re
import pandas
import os
from sklearn.model_selection import KFold, ShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, balanced_accuracy_score, precision_score, recall_score, f1_score, log_loss
from sklearn.decomposition import PCA


def transform_(x):
    if isinstance(x, str):
        digits = re.findall("\d", x)
        if len(digits) == 0:
            return None
        else:
            return int(digits[0])
    else:
        return None




def load_features(path_, type_, patient_id, columns_to_keep=["oar", "name", "value"]):
    """
    load features

    args:
        path_ (str) path to features folder
        type_ (str) features type (radiomics, dosiomics, dvh, deepnn)
        patient_id (int,str) id of patient to find its folder
        columns_to_keep (list(str)) list of columns name to keep
    """
    patient_folder = os.path.join(path_, [i for i in os.listdir(path_) if int(i) == int(patient_id)][0])
    features = pandas.read_csv(os.path.join(patient_folder, f"{type_}.csv"))
    
    # if "Unnamed: 0" in features.columns:
    #     features = features.drop(columns=["Unnamed: 0"])
    
    if columns_to_keep:
        features = features[columns_to_keep]
    
    return features

def load_radiomics(path_, patient_id, omics):
    """
    omics (radiomics, dosiomics)
    """
    assert omics in ["radiomics", "dosiomics"]
    df = load_features(path_, omics, patient_id, columns_to_keep=None)
    df = df[(df["type"] == "original")]
    df['name'] = df[["type", "class", "name"]].apply(lambda row: '_'.join(row.values.astype(str)), axis=1)
    return df.drop(columns=["type", "class"])




def get_unique_combos(df, columns):
    unique_combos = df[[*columns]].drop_duplicates()
    return list(unique_combos.itertuples(index=False, name=None))




def build_dataset(cohort_path, features_path, combined_path):
    # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!! adapt this for TCIA 
    clinical = ['SEX', 'AGE', 'ST_STAG', 'ECOG']
    CLINICAL_MAPPING = {
        # SEX
        "Male": 1,
        "Female": 2,
        # "M": 1,
        # "F": 2,

        # CANCER STAGING
        "Stage III": 1,
        "Stage IVa": 2,
        "Stage IVb": 3,

        # ECOG
        "Asympatomatic": 0,
        "Completely ambulatory": 1,
        "lower than 50% in bed": 2,
    }

    print("loading cohort...")
    with open(cohort_path, "rb") as f:
        patients = pickle.load(f)

    print("building dataset...")
    df = []
    for p in patients:
        patient_df = []

        for k in clinical:
            try:
                patient_df.append({"features": "clinical", "name": k, "value": int(p.clinical[k]), "id": int(p.id)})
            except ValueError:
                patient_df.append({"features": "clinical", "name": k, "value": CLINICAL_MAPPING[p.clinical[k]], "id": int(p.id)})

        # select dysphagia and xerostomia in CTCAE
        # transform values
        # change S0 into Inclusion
        # if patient has baseline tox, skip it
        # keep only M6 grade
        cm = pandas.DataFrame(p.clinical_measurements)
        cm = cm[(cm["type"] == "AE") & (cm["sample"].isin(["XEROSTOMIE", "DYSPHAGIE"]))]
        cm["value"] = cm["value"].apply(transform_)
        cm.loc[(cm["visitID"] == "S0"), "visitID"] = "Inclusion"
        if (cm[(cm["visitID"] == "Inclusion") & (cm["sample"] == "XEROSTOMIE")]["value"].astype('Int64') > 0).any():
            continue
        cm = cm[cm["visitID"] == "M6"]

        # transform tox value to binary
        cm.loc[(cm["value"] < 2), "value"] = 0
        cm.loc[(cm["value"] >= 2), "value"] = 1

        for _, row in cm.iterrows():
            patient_df.append({"features": "clinical", "name": row["sample"], "value": row["value"], "id": int(p.id)})

        try:
            # radiomics
            radiomics = load_radiomics(features_path, p.id, "radiomics")
            radiomics["id"] = int(p.id)
            radiomics["features"] = "radiomics"
            patient_df.extend(radiomics.to_dict(orient="records"))
        except (FileNotFoundError, IndexError):
            print(f"WARNING: patient {p.id} missing radiomics data")
            continue

        try:
            # dosiomics
            dosiomics = load_radiomics(features_path, p.id, "dosiomics")
            dosiomics["id"] = int(p.id)
            dosiomics["features"] = "dosiomics"
            patient_df.extend(dosiomics.to_dict(orient="records"))
        except (FileNotFoundError, IndexError):
            print(f"WARNING: patient {p.id} missing dosiomics data")
            continue

        try:
            # dvh
            dvh = load_features(features_path, "dvh", p.id)
            dvh["id"] = int(p.id)
            dvh["features"] = "dvh"
            patient_df.extend(dvh.to_dict(orient="records"))
        except (FileNotFoundError, IndexError):
            print(f"WARNING: patient {p.id} missing dvh data")
            continue

        try:
            # deepnn
            deepnn = load_features(features_path, "deepnn", p.id)
            deepnn["id"] = int(p.id)
            deepnn["features"] = "deepnn"
            patient_df.extend(deepnn.to_dict(orient="records"))
        except (FileNotFoundError, IndexError):
            print(f"WARNING: patient {p.id} missing deepnn data")
            continue

        df.extend(patient_df)

    print("saving dataset...")
    pandas.DataFrame(df).to_csv(combined_path, index=False)



def display_split_stats(split):
    stats = {j: list(split).count(j) for j in set(split)}
    for j, c in stats.items():
        print(f"\t {j}: {c} ({int(100*c/len(split))}%)")

def cross_validation(X, Y, kfold=3, bootstrap=None, feature_selection=None, feature_reduction_N=None):
    """
    Perform cross validation training and return classification metrics as dict

    args
        X (numpy.ndarray) training data input of shape (n_samples, n_features)
        Y (numpy.ndarray) training data expected output of shape (n_samples)
        kfold (int) number of folds default is 3
        bootstrap (int) number of time to eprform the training with random splitting, if None apply k-fold
        feature_selection (str) feature selection method to apply (see sklearn)
        feature_reduction_N (int) number of dimension to reduce data into using PCA
    """

    if bootstrap:
        splitter = ShuffleSplit(n_splits=bootstrap, random_state=0, test_size=0.3)
    else:
        splitter = KFold(n_splits=kfold)

    metrics = []
    reductor = []
    for i, (train_index, test_index) in enumerate(splitter.split(X)):
        x_train, y_train = X[train_index], Y[train_index]
        x_test, y_test = X[test_index], Y[test_index]

        # print("fold ", i+1)
        # print("train:")
        # display_split_stats(y_train)
        # print("test:")
        # display_split_stats(y_test)

        if feature_selection:
            raise NotImplementedError() #TODO

        if feature_reduction_N:
            try:
                # print(f"transforming input from {x_train.shape[1]} to {feature_reduction_N} dimensions..")
                reduc = PCA(n_components=feature_reduction_N, random_state=0)
                reduc.fit(x_train)
                x_train = reduc.transform(x_train)
                x_test = reduc.transform(x_test)
                reductor.append(reduc)
            except ValueError:
                print("ValueError occured during data dimension reduction")
                continue
        
        try:
            clf = LogisticRegression(random_state=0, verbose=0)
            clf.fit(x_train, y_train)
            y_pred = clf.predict(x_test)
            y_pred_proba = clf.predict_proba(x_test)
            metrics.append({
                "acc": accuracy_score(y_test, y_pred),
                "auc": roc_auc_score(y_test, y_pred),
                "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
                "precision": precision_score(y_test, y_pred, zero_division=0),
                "recall": recall_score(y_test, y_pred, zero_division=0),
                "f1_score": f1_score(y_test, y_pred, zero_division=0),
                "log_loss": log_loss(y_test, y_pred_proba),
            })
        except ValueError:
            print("ValueError occured during model training or test")
            continue

    return metrics, reductor

In [2]:
import os, json
import numpy as np
import tqdm
from sklearn.preprocessing import scale, minmax_scale

def run_experiment(cohort_path, features_path, exps, oars, exp_code, feature_reduction_N, normalization, bootstrap=None, kfold=3):
    combined_path = "./dataset.csv"
    build_dataset(cohort_path, features_path, combined_path)
    df = pandas.read_csv(combined_path)

    print("filtering patients...")
    # filter patients without xerostomia label
    # add patients without any XEROSTOMIA line
    subdf = df[df["name"] == "XEROSTOMIE"]
    subdf = subdf[pandas.isna(subdf["value"])]
    patients_to_exclude = list(subdf["id"].unique())
    subdf = df[df["name"] == "XEROSTOMIE"]
    patients_to_exclude.extend(list(set(set(df["id"].unique()).difference(subdf["id"].unique()))))

    print("patients excluded due to absent xerostomia label:")
    print(patients_to_exclude)
    df = df[~df["id"].isin(patients_to_exclude)]

    for i, exp_params in enumerate(tqdm.tqdm(exps)):
        for oar in oars:
            # create experience name
            exp_name = f"{exp_code}_{oar}_{str(i+1)}"

            # select data containing OARs without removing clinical data (contain label)
            exp_df = df[(df["oar"] == oar) | (df["features"] == "clinical")]

            # select features
            for fts, names in exp_params.items():
                if names == -1:
                    # keep all features belong to this type
                    pass
                elif isinstance(names, list):
                        exp_df = exp_df.drop(exp_df[(exp_df["features"] == fts) & (~exp_df["name"].isin(names))].index)
                else:
                    raise TypeError()
                
            print("features:")
            features = get_unique_combos(exp_df, ("features", "name"))
            print(features)

            # build X,Y
            patients = exp_df["id"].unique()
            X = [[exp_df[(exp_df["id"] == int(p)) & (exp_df["features"] == fts) & (exp_df["name"] == name_)]["value"].item() for fts, name_ in features] for p in patients]
            # !!! this must be done on original DataFrame because txo value is filtered in exp dataframe
            Y = [df[(df["id"] == int(p)) & (df["name"] == "XEROSTOMIE")]["value"].item() for p in patients]
            X = np.array(X, dtype=np.float32)
            Y = np.array(Y, dtype=np.float16).astype(dtype=np.int16)

            # normalize features
            if normalization == "minmax":
                X = minmax_scale(X, axis=0)
            else:
                X = scale(X, axis=0)

            # fit model
            if bootstrap:
                metrics, _ = cross_validation(X, Y, bootstrap=bootstrap, feature_reduction_N=feature_reduction_N)
            else:
                metrics, _ = cross_validation(X, Y, kfold=kfold, feature_reduction_N=feature_reduction_N)

            # save results
            out_dir = os.path.join("./experiments", exp_code, exp_name)
            os.makedirs(out_dir, exist_ok=True)
            pandas.DataFrame(metrics).to_csv(os.path.join(out_dir, "metrics.csv"))

            # save exp params
            with open(os.path.join(out_dir, "params.json"), "w") as f:
                save_params = {**exp_params, "features": features, "feature_reduction_N": feature_reduction_N, 
                            "normalization": normalization, "oar": oar, "bootstrap": bootstrap, "kfold": kfold}
                json.dump(save_params, f)

In [20]:
from radiomics import getFeatureClasses

def list_radiomics(type_):
    features = getFeatureClasses()[type_].getFeatureNames().keys()
    return [f"original_{type_}_{i}" for i in features]

exp_code = "005"
feature_reduction_N = None
normalization = "minmax"
bootstrap = 100
kfold = 3
cohort_path = r"C:\Users\bilel.guetarni\Desktop\ARTIX\data\artix.pkl"

# oars = ["parotid_gland_ipsi", "parotid_gland_contra"]
# features_path = r"C:\Users\bilel.guetarni\Desktop\ARTIX\features\original\artix"

oars = ["parotid_gland_left", "parotid_gland_right"]
features_path = r"C:\Users\bilel.guetarni\Desktop\ARTIX\features\totalsegmentator\artix"

exps = [
    {"clinical": [], 
     "radiomics": [],
     "dosiomics": ["original_firstorder_Mean"], 
     "dvh": [], 
     "deepnn": []},
]

run_experiment(cohort_path, features_path, exps, oars, exp_code, feature_reduction_N, normalization, bootstrap=bootstrap, kfold=kfold)

loading cohort...
building dataset...
saving dataset...
filtering patients...
patients excluded due to absent xerostomia label:
[63, 59, 44, 29, 128, 107, 1, 25, 53, 113, 123]


  0%|          | 0/1 [00:00<?, ?it/s]

features:
[('dosiomics', 'original_firstorder_Mean')]
features:
[('dosiomics', 'original_firstorder_Mean')]


100%|██████████| 1/1 [00:04<00:00,  4.48s/it]


# Analysis

In [21]:
import os
import pandas
import plotly.express as px

path_ = r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments"

for exp in os.listdir(path_):
    result_df = []
    for run in os.listdir(os.path.join(path_, exp)):
        metrics_df = pandas.read_csv(os.path.join(path_, exp, run, "metrics.csv"))
        metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
        avg_metrics = metrics_df.mean().to_dict()
        for k, v in avg_metrics.items():
            result_df.append({"exp": run, "metric": k, "value": v, "std": metrics_df.std().to_dict()[k]})

    print(len(result_df))
    result_df = pandas.DataFrame(result_df)

    # for metric in result_df["metric"].unique():
    #     metric_df = result_df[result_df["metric"] == metric]
    #     fig = px.bar(metric_df, x="run", y="value", error_y ="std", title=metric)
    #     fig.show()

    fig = px.bar(result_df, x="exp", y="value", color="metric", barmode="group", title=exp)
    # fig = px.bar(result_df, x="exp", y="value", error_y ="std", color="metric", barmode="group", title=exp)
    fig.show()

70


70


14


14


In [3]:
import os, json
import pandas
import plotly.express as px

path_ = r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\004"
result_df = []
for run in os.listdir(path_):
    with open(os.path.join(path_, run, "params.json"), "r") as f:
        params = json.load(f)

    metrics_df = pandas.read_csv(os.path.join(path_, run, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        result_df.append({"oar": params["oar"], "metric": k, "value": v, "std": metrics_df.std().to_dict()[k]})

result_df = pandas.DataFrame(result_df)

fig = px.bar(result_df, x="oar", y="value", color="metric", barmode="group")
fig.update_layout(autosize=False, width=800, height=400)
fig.show()
fig.write_image(r"C:\Users\bilel.guetarni\Downloads\newplot.png")